In [ ]:
import torch.utils.data
from hydra import initialize, compose
from mlflow.gateway.providers import gemini
from omegaconf import DictConfig
import os

from sentence_transformers.util import similarity

from src.data.dataloader import data_loader
from src.data.dataset_aprdb import AprdbDataset

import src.config
from src.detector.cristal_anchor.fetch_academic_paper import extract_text_from_pdf

with initialize(version_base=None, config_path="../conf"):
    cfg: DictConfig = compose(config_name="config", overrides=["+experiment/dataset/aprbd@dataset=aprdb_tt_test_all",
                                                               "experiment/data_loader=1_4_true"])

os.environ['CUDA_LAUNCH_BLOCKING'] = "1"

print(cfg)


In [ ]:
ds_test = AprdbDataset(config=cfg.dataset)
len(ds_test)


In [ ]:
import pandas as pd

rows = []
for i in range(len(ds_test)):
    item = ds_test[i]
    rows.append(item)
df = pd.DataFrame(rows)
df


In [ ]:
df_generated = pd.read_json('../data/ai-peer-review-detection-benchmark/reviews.json')
print(df_generated.head())
df_generated = df_generated[['label', 'generated_review']]
df_generated['generator'] = "mgt"
df_generated.rename(columns={'generated_review': 'content'}, inplace=True)

# df_generated_1 = df_generated[df_generated['label'][0] == 1]]

# df_0_mean = df_generated_0['similarity'].mean()
# df_1_mean = df_generated_1['similarity'].mean()


In [ ]:
df

In [ ]:
df_generated

In [ ]:
from sentence_transformers import SentenceTransformer

embedding_model = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")

## test embedding
embedded_target = embedding_model.encode("test1")
embedded_generated_review = embedding_model.encode("test2")

similarity = embedding_model.similarity(embedded_target, embedded_generated_review)
print(similarity)



In [ ]:
df

In [ ]:
hwf_only_df = df[df['label'] == 0]
mgt_only_df = df[df['label'] == 1]
hwf_only_df = hwf_only_df.copy()

In [ ]:
mgt_only_df.generator.unique()

In [ ]:
claude_df = mgt_only_df[mgt_only_df['generator'] == 'claude']
gpt4o_df = mgt_only_df[mgt_only_df['generator'] == 'gpt4o']
gemini_df = mgt_only_df[mgt_only_df['generator'] == 'gemini']
llama_df = mgt_only_df[mgt_only_df['generator'] == 'llama']
qwen_df = mgt_only_df[mgt_only_df['generator'] == 'qwen']
qwen_3_generated = df_generated[df_generated['generator'] == 'mgt']


In [ ]:
qwen_df

In [ ]:
hwf_only_df['generator'] = "human"

In [ ]:
print("claude", claude_df.generator.unique())
print("gemini", gemini_df.generator.unique())
print("generated_qwen3", qwen_3_generated.generator.unique())


In [ ]:
from itertools import combinations

df_all_list = list(combinations([hwf_only_df, gemini_df, llama_df, qwen_df, gpt4o_df, claude_df, qwen_3_generated], 2))

In [ ]:
def compare_similarity(a, b):
    embedded_a = embedding_model.encode(a, show_progress_bar=False)
    embedded_b = embedding_model.encode(b, show_progress_bar=False)
    similarity = embedding_model.similarity(embedded_a, embedded_b)
    return similarity


In [ ]:
import tqdm

values_similarity = {}

for df_1, df_2 in tqdm.tqdm(df_all_list, total=len(df_all_list)):
    mean_similarities = 0
    count = 0
    print(df_1['generator'].unique(), df_2['generator'].unique())
    results = [ compare_similarity(a, b) for a, b in zip(df_1['content'], df_2['content'])]
    mean = sum(results) / len(results)
    values_similarity[(df_1['generator'].unique()[0], df_2['generator'].unique()[0]) ] = mean



In [ ]:
import json

def to_serializable(x):
    if isinstance(x, torch.Tensor):
        return x.detach().cpu().tolist()
    return x

infos_clean = [{k: to_serializable(v) for k, v in d.items()} for d in values_similarity]
with open("similarities.json", "w", encoding="utf-8") as f:
    json.dump(infos_clean, f, ensure_ascii=False, indent=2)


## evaluating results

In [ ]:
import pandas as pd
df_results = pd.read_json("../data/ai-peer-review-detection-benchmark/similarities_2.json")
df_results